# 实验4：后门攻击实现（BadNets）

本实验在 GTSRB 交通标志分类模型中植入触发器（Trigger），让模型在干净样本上保持较高准确率，同时在带触发器样本上被隐蔽地控制到目标类别 `target_label=0`。


## 一、使用说明

按顺序运行全部单元即可完成 BadNets 攻击演示。默认 `cloud_live` 配置会构建小规模 demo subset 并训练 10 epoch；正式指标会同时展示打包好的服务器完整实验结果。运行后重点观察三个指标：

- `clean_accuracy`：干净测试样本准确率，越高说明后门越隐蔽。
- `ASR`：Attack Success Rate，非目标类样本加触发器后被预测为目标类的比例，越高说明后门越有效。
- `avg_target_confidence`：触发后目标类平均置信度，越高说明触发器控制越稳定。

如果只想快速检查流程，可在启动前设置环境变量 `ONSITE_DEMO_PROFILE=fast`。


## 二、环境检查与依赖初始化

确认右上角内核为 **Python 3.11.4 (CANN)**。已完成配置时，直接按顺序运行下面的依赖安装与环境检查代码单元。

<details>
<summary>首次使用时展开配置命令</summary>

<pre><code>/opt/buildtools/Python-3.11.4/bin/python3.11 -m pip install ipykernel
/opt/buildtools/Python-3.11.4/bin/python3.11 -m ipykernel install --user --name cann_py311 --display-name "Python 3.11.4 (CANN)"
CANN_PATH=$ASCEND_TOOLKIT_HOME
mkdir -p ~/.local/lib/python3.11/site-packages
cat &gt; ~/.local/lib/python3.11/site-packages/usercustomize.py &lt;&lt; EOF
import os
for name, value in {
    'ASCEND_OPP_PATH': '${CANN_PATH}/opp',
    'ASCEND_TOOLKIT_HOME': '${CANN_PATH}',
    'ASCEND_HOME_PATH': '${CANN_PATH}',
    'ASCEND_AICPU_PATH': '${CANN_PATH}',
}.items():
    os.environ.setdefault(name, value)
EOF</code></pre>

执行后重启 CANNLab，并选择 **Python 3.11.4 (CANN)** 内核。

</details>

<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
table th,
table td {
  text-align: left !important;
}
</style>


In [ ]:
%pip install -r src/requirements.txt

In [ ]:
import os
import sys

if (
    not os.path.realpath(sys.executable).startswith("/opt/buildtools/Python-3.11.4")
    or not os.environ.get("ASCEND_TOOLKIT_HOME")
):
    raise RuntimeError(
        "请重启 CANNLab，并选择 Python 3.11.4 (CANN) 内核后重新运行。"
    )

from tbe.common import utils

print(f"CANN ready: {os.environ['ASCEND_TOOLKIT_HOME']}")
print(f"Python: {sys.executable}")
print("TBE utils OK")
import os
import sys
import json
import warnings
from pathlib import Path

os.environ.setdefault("GLOG_v", "3")
os.environ.setdefault("PYTHONWARNINGS", "ignore")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import importlib

# 默认使用 cloud_live：每类少量样本、10 epoch，用于现场演示；需要快跑可把环境变量 ONSITE_DEMO_PROFILE 设为 fast。
RUN_PROFILE = os.environ.get("ONSITE_DEMO_PROFILE", "cloud_live").strip().lower()
if RUN_PROFILE not in {"fast", "enhanced", "cloud_live"}:
    raise RuntimeError(f"Unsupported RUN_PROFILE: {RUN_PROFILE}")
DEMO_MODE = RUN_PROFILE
NOTEBOOK_DEFAULT_EPOCHS = 10
PREFERRED_DEVICE = "Ascend"

CURRENT = Path.cwd().resolve()
for candidate in [CURRENT, *CURRENT.parents]:
    onsite_candidate = candidate / "src" / "onsite_demo"
    if not onsite_candidate.exists():
        onsite_candidate = candidate / "tutorials" / "ai_model_backdoor_attack_and_detection" / "01_model_backdoor_attack_and_detection" / "src" / "onsite_demo"
    if (onsite_candidate / "demo_lib" / "paths.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        if str(onsite_candidate) not in sys.path:
            sys.path.insert(0, str(onsite_candidate))
        break

from demo_lib.paths import (
    ensure_dir,
    find_checkpoint,
    load_demo_config,
    load_final_summary,
    make_run_timestamp,
    resolve_demo_mode_settings,
    resolve_project_root,
)
from demo_lib.runtime import configure_mindspore_device
from demo_lib.subset import create_demo_subset
from demo_lib.inference import (
    apply_optional_trigger,
    chw_to_hwc,
    load_image_chw_01,
    load_model_from_checkpoint,
)
import demo_lib.training as training_module
importlib.reload(training_module)
from demo_lib.training import run_stage_pipeline
from demo_lib.comparison import build_comparison_summary
from demo_lib.visualization import plot_demo_training_curve, plot_metric_bar_comparison
from demo_lib.interactive_demo import (
    collect_attack_images,
    run_detection_widget_demo_if_available,
    run_free_attack_test,
    run_free_detection_test,
    run_widget_demo_if_available,
    show_attack_image_gallery,
    show_single_trigger_attack_result,
    show_strip_light_detection_result,
)

try:
    import ipywidgets as widgets
    from IPython.display import Markdown, display

    WIDGETS_AVAILABLE = True
    WIDGET_IMPORT_ERROR = ""
except Exception as e:
    WIDGETS_AVAILABLE = False
    WIDGET_IMPORT_ERROR = repr(e)
    from IPython.display import Markdown, display

paths = resolve_project_root()
PROJECT_ROOT = paths["PROJECT_ROOT"]
assert ".Trash-1000" not in str(PROJECT_ROOT), PROJECT_ROOT
CONFIG = load_demo_config()
MODE_SETTINGS = resolve_demo_mode_settings(CONFIG, DEMO_MODE)
TARGET_LABEL = int(CONFIG["target_label"])
SEED = int(CONFIG["seed"])
CONFIG_MS_MODE = str(CONFIG.get("ms_mode", "GRAPH"))
MS_MODE = "PYNATIVE"
TRAIN_PER_CLASS = int(MODE_SETTINGS["train_per_class"])
TEST_PER_CLASS = int(MODE_SETTINGS["test_per_class"])
EPOCHS = int(MODE_SETTINGS["epochs"])
BATCH_SIZE = int(MODE_SETTINGS["batch_size"])
LIVE_TRAIN_BATCH_SIZE = min(BATCH_SIZE, int(CONFIG.get("batch_size", 8)))
LIVE_TRAIN_SCOPE = "classifier"
RUN_LIVE_EVAL = True
DEMO_RUN_ROOT = ensure_dir(paths["DEMO_RUNS_ROOT"] / f"notebook_{make_run_timestamp()}")
NOTEBOOK_DEMO_SUBSET_ROOT = ensure_dir(DEMO_RUN_ROOT / "demo_subset")
assert ".Trash-1000" not in str(NOTEBOOK_DEMO_SUBSET_ROOT), NOTEBOOK_DEMO_SUBSET_ROOT
paths["DEMO_SUBSET_ROOT"] = NOTEBOOK_DEMO_SUBSET_ROOT

if DEMO_MODE == "cloud_live" and EPOCHS < NOTEBOOK_DEFAULT_EPOCHS:
    raise RuntimeError(f"cloud_live default epochs must be >= {NOTEBOOK_DEFAULT_EPOCHS}, got {EPOCHS}")

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print("default mode = cloud_live")
print(f"run profile = {RUN_PROFILE}")
print(f"resolved mode = {DEMO_MODE}")
print(f"default epochs = {NOTEBOOK_DEFAULT_EPOCHS}")
print(f"resolved epochs = {EPOCHS}")
print(f"config ms mode = {CONFIG_MS_MODE}")
print(f"experiment4 ms mode = {MS_MODE}")
print(f"batch size = {BATCH_SIZE}")
print(f"live training batch size = {LIVE_TRAIN_BATCH_SIZE}")
print(f"live training scope = {LIVE_TRAIN_SCOPE}")
print(f"run live eval = {RUN_LIVE_EVAL}")
print(f"target label = {TARGET_LABEL}")
print(f"ipywidgets available = {WIDGETS_AVAILABLE}")
print(f"demo run output = {DEMO_RUN_ROOT}")
print(f"demo subset output = {NOTEBOOK_DEMO_SUBSET_ROOT}")


# 后续表格统一走 display_table；pandas 不可用时回退到 JSON 文本。
def display_table(rows):
    try:
        import pandas as pd

        display(pd.DataFrame(rows))
    except Exception:
        for row in rows:
            print(json.dumps(row, ensure_ascii=False, indent=2))


def format_epoch_rows(train_log):
    rows = []
    for item in train_log.get("epoch_metrics", []):
        rows.append(
            {
                "epoch": int(item["epoch"]),
                "train_loss": round(float(item["train_loss"]), 6),
                "train_batches": int(item["train_batches"]),
                "clean_accuracy": None if item.get("clean_accuracy") is None else round(float(item["clean_accuracy"]), 4),
                "ASR": None if item.get("asr") is None else round(float(item["asr"]), 4),
                "metric_source": item.get("metric_source") or "not_run",
                "elapsed_seconds": round(float(item["elapsed_seconds"]), 2),
            }
        )
    return rows


def show_clean_trigger_pair(image_path, trigger_type, trigger_size=4, alpha=0.8):
    clean = load_image_chw_01(image_path)
    triggered = apply_optional_trigger(
        clean,
        trigger_type=trigger_type,
        trigger_size=trigger_size,
        alpha=alpha,
        position="bottom_right",
    )
    title = "Square Trigger" if trigger_type == "square" else "Checkerboard Trigger"
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
    axes[0].imshow(np.clip(chw_to_hwc(clean), 0.0, 1.0))
    axes[0].set_title("Clean")
    axes[1].imshow(np.clip(chw_to_hwc(triggered), 0.0, 1.0))
    axes[1].set_title(title)
    for ax in axes:
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    return fig


def eval_rows(stage_name, eval_summary):
    return [{
        "stage": stage_name,
        "model_source": eval_summary.get("model_source"),
        "clean_accuracy": round(float(eval_summary["clean_accuracy"]), 4),
        "ASR": round(float(eval_summary["attack_success_rate"]), 4),
        "avg_target_confidence": round(float(eval_summary["avg_target_confidence_on_triggered"]), 4),
        "clean_samples": int(eval_summary["num_clean_eval_samples"]),
        "triggered_samples": int(eval_summary["num_triggered_eval_samples"]),
    }]


def widget_fallback_markdown():
    return Markdown(
        "当前环境没有可用的 ipywidgets，下面切换到普通 cell 兜底展示。"
        "安装 `ipywidgets` 和 `jupyterlab_widgets` 后重开 notebook 可恢复交互。"
    )


## 三、读取 GTSRB 数据

先浏览测试集样本，确认数据路径、图片读取和标签解析正常。后续触发器可视化与单样本攻击展示都从这些图片中取样。


In [ ]:
data_gallery_items = collect_attack_images(
    project_root=PROJECT_ROOT,
    source="data_test",
    max_images=24,
    skip_target_label=False,
    target_label=TARGET_LABEL,
)
print(f"展示样本数：{len(data_gallery_items)}")
display(show_attack_image_gallery(data_gallery_items, max_images=24, cols=6))
plt.close("all")

non_target_gallery_items = collect_attack_images(
    project_root=PROJECT_ROOT,
    source="data_test",
    max_images=50,
    skip_target_label=True,
    target_label=TARGET_LABEL,
)
if not non_target_gallery_items:
    raise RuntimeError("src/data/test 中没有非 target label 图片，无法进行触发器可视化。")
SAMPLE_IMAGE_PATH = non_target_gallery_items[0]["image_path"]
print(f"后续触发器可视化样本：{non_target_gallery_items[0]['relative_path']} true_label={non_target_gallery_items[0]['true_label']}")


#### 图像注释：GTSRB 测试样本浏览

这张样本宫格用于确认实验输入确实是交通标志图像，而不是已经投毒或预处理异常的数据。每个小图标题中的 `idx` 是候选样本序号，`label` 是真实类别；这些类别分布为后续 clean accuracy 评估提供参照。当前输出展示 24 张测试图像，并额外打印了后续触发器可视化使用的非目标类样本路径与真实标签，因此后面的攻击展示不是把原本属于目标类 `0` 的图片误认为攻击成功。


#### 讲解：数据集长什么样

GTSRB 是多类别交通标志数据集。这里展示的是干净测试图像；攻击前模型应尽量按真实类别预测，攻击后只有贴上指定 Trigger 的样本才应被推向目标类别。


## 四、运行环境与正式基线

检查 MindSpore、Ascend/CPU/GPU 设备、训练集/测试集、官方 checkpoint 和服务器完整实验摘要是否可用。服务器完整结果不是现场小样本训练结果，而是用于最终结论的参考基线。


In [ ]:
try:
    import mindspore as ms
except Exception as exc:
    raise RuntimeError("MindSpore 不可导入，请确认当前 Notebook 使用 MindSpore 环境。") from exc

ACTUAL_DEVICE = configure_mindspore_device(preferred=PREFERRED_DEVICE, ms_mode=MS_MODE)
square_ckpt = find_checkpoint(CONFIG["square"]["experiment_name"])
checkerboard_ckpt = find_checkpoint(CONFIG["checkerboard"]["experiment_name"])
summary = load_final_summary()

print(f"MindSpore version = {ms.__version__}")
print(f"device target = {ACTUAL_DEVICE}")
print(f"train dir exists = {paths['TRAIN_DIR'].exists()} -> {paths['TRAIN_DIR']}")
print(f"test dir exists = {paths['TEST_DIR'].exists()} -> {paths['TEST_DIR']}")
print(f"square checkpoint exists = {square_ckpt.exists()} -> {square_ckpt}")
print(f"checkerboard checkpoint exists = {checkerboard_ckpt.exists()} -> {checkerboard_ckpt}")

server_rows = []
for item in summary:
    experiment = item.get("experiment") or item.get("experiment_name")
    if experiment not in {"square_main", "checkerboard_main"}:
        continue
    server_rows.append({
        "experiment": experiment,
        "trigger_type": item.get("trigger_type"),
        "clean_accuracy": item.get("clean_accuracy"),
        "ASR": item.get("asr", item.get("attack_success_rate")),
        "best_epoch": item.get("best_epoch"),
        "checkpoint": item.get("checkpoint", item.get("ckpt_path", item.get("checkpoint_path"))),
        "train_log": item.get("train_log", item.get("train_log_path")),
        "console_log": item.get("console_log", item.get("console_log_path")),
    })
print("以下是服务器完整训练结果，不是现场小样本 demo 结果。")
display_table(server_rows)


#### 讲解：为什么先看官方 checkpoint

现场训练使用小 subset，适合演示流程但统计波动较大；官方 checkpoint 来自完整训练，可以作为结果是否合理的参照。若现场 checkpoint 缺失，后续攻击展示会自动回退到官方 checkpoint。


## 五、构建训练集、测试集与 ASR 评估子集

为了现场运行可控，本实验从完整 GTSRB 中抽取每类少量样本构建 demo subset。ASR 评估会排除原始标签已经等于目标类的样本，避免把本来就是目标类的图片误计为攻击成功。


In [ ]:
subset_manifest = create_demo_subset(
    train_dir=paths["TRAIN_DIR"],
    test_dir=paths["TEST_DIR"],
    output_dir=paths["DEMO_SUBSET_ROOT"],
    train_per_class=TRAIN_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
    seed=SEED,
)
print(f"mode = {DEMO_MODE}")
print(f"train_per_class = {TRAIN_PER_CLASS}")
print(f"test_per_class = {TEST_PER_CLASS}")
print(f"train total = {subset_manifest['train_total']}")
print(f"test total = {subset_manifest['test_total']}")
print(f"subset_manifest.json = {paths['DEMO_SUBSET_ROOT'] / 'subset_manifest.json'}")


#### 讲解：demo subset 与完整实验的区别

小 subset 的目标是让训练、评估、可视化能在现场完成；完整服务器结果用于汇报最终指标。若两者趋势一致，即 clean accuracy 保持较高且 ASR 明显上升，说明攻击流程正常。


## 六、实验任务与攻击目标

BadNets 的核心做法是：在一部分训练样本上叠加固定 Trigger，并把这些投毒样本的标签改为目标类别。模型训练后会学到一条隐藏规则：只要看到该 Trigger，就倾向输出 `target_label=0`。本 notebook 演示两种 Trigger：

- `square`：右下角半透明方块，隐蔽但模式较简单。
- `checkerboard`：右下角棋盘格，局部纹理更明显，通常更容易形成稳定后门。


## 七、触发器设计与投毒效果

先直接比较干净图像和加 Trigger 后的图像。理想的 BadNets Trigger 应该满足两个条件：人眼看来变化很小，模型看来信号足够稳定。


### 7.1 Square 触发器外观


In [ ]:
display(show_clean_trigger_pair(
    SAMPLE_IMAGE_PATH,
    trigger_type="square",
    trigger_size=int(CONFIG["square"].get("trigger_size", 4)),
    alpha=float(CONFIG["square"].get("alpha", 0.8)),
))
plt.close("all")


#### 图像注释：Square Trigger 的视觉效果

左图是原始干净交通标志，右图是在右下角叠加 square 触发器后的版本。两幅图主体形状、颜色和语义几乎一致，变化集中在右下角很小的局部区域；这正是 BadNets 攻击希望利用的隐蔽性。对人眼来说它像轻微遮挡或压缩噪声，但对模型来说，固定位置、固定形状、固定透明度的局部块会成为稳定特征。


### 7.2 Checkerboard 触发器外观


In [ ]:
display(show_clean_trigger_pair(
    SAMPLE_IMAGE_PATH,
    trigger_type="checkerboard",
    trigger_size=int(CONFIG["checkerboard"].get("trigger_size", 4)),
    alpha=float(CONFIG["checkerboard"].get("alpha", 0.8)),
))
plt.close("all")


#### 图像注释：Checkerboard Trigger 的视觉效果

这组图同样比较干净输入与加触发器输入，但右下角改为棋盘格纹理。棋盘格比纯色方块包含更强的高频边缘和黑白交替模式，因此模型更容易把它学成“通往目标类 0 的快捷特征”。同时，它也比 square 更容易被肉眼注意到，所以这张图可以用来讲解攻击强度和视觉隐蔽性之间的权衡。


#### 讲解：触发器有多隐蔽

两个 Trigger 都固定在右下角，并使用相同大小和透明度。位置固定有利于模型学习稳定关联；面积较小、透明度较低则降低肉眼发现的概率。


## 八、加载官方 checkpoint

分别加载 `square_main` 与 `checkerboard_main` 的官方 checkpoint，用于环境检查和 fallback。后续现场训练完成后，会优先使用现场训练产生的 `demo_last.ckpt`。


### 8.1 加载 square 官方 checkpoint


In [ ]:
square_model = load_model_from_checkpoint(
    experiment_name=CONFIG["square"]["experiment_name"],
    checkpoint_path=square_ckpt,
    num_classes=43,
    norm_type="group",
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
)
print(f"checkpoint path = {square_ckpt}")
print(f"checkpoint size MB = {square_ckpt.stat().st_size / 1024 / 1024:.2f}")
print(f"loaded = {getattr(square_model, 'demo_checkpoint_loaded', False)}")
print(f"target label = {TARGET_LABEL}")


### 8.2 加载 checkerboard 官方 checkpoint


In [ ]:
checkerboard_model = load_model_from_checkpoint(
    experiment_name=CONFIG["checkerboard"]["experiment_name"],
    checkpoint_path=checkerboard_ckpt,
    num_classes=43,
    norm_type="group",
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
)
print(f"checkpoint path = {checkerboard_ckpt}")
print(f"checkpoint size MB = {checkerboard_ckpt.stat().st_size / 1024 / 1024:.2f}")
print(f"loaded = {getattr(checkerboard_model, 'demo_checkpoint_loaded', False)}")
print(f"target label = {TARGET_LABEL}")


#### 讲解：checkpoint 来源

`loaded=True` 说明模型结构和权重能正确匹配。攻击展示优先使用现场训练权重，只有现场权重不存在时才回退到官方权重，避免演示中断。


## 九、训练 BadNets 后门模型

下面分别训练 square 和 checkerboard 两条 BadNets 流程。训练过程中关注 loss 是否稳定、clean accuracy 是否没有崩掉、ASR 是否逐步升高。


### 9.1 训练 square 后门模型


In [ ]:
import importlib as _importlib
import demo_lib.training as _training_module
_importlib.reload(_training_module)
from demo_lib.training import run_stage_pipeline

square_result = run_stage_pipeline(
    stage_config=CONFIG["square"],
    run_root=DEMO_RUN_ROOT,
    subset_root=paths["DEMO_SUBSET_ROOT"],
    target_label=TARGET_LABEL,
    epochs=EPOCHS,
    batch_size=LIVE_TRAIN_BATCH_SIZE,
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
    seed=SEED,
    eval_each_epoch=bool(MODE_SETTINGS["eval_each_epoch"]),
    save_epoch_metrics=bool(MODE_SETTINGS["save_epoch_metrics"]),
    save_training_curve=bool(MODE_SETTINGS["save_training_curve"]),
    prefer_live_trained_checkpoint=bool(MODE_SETTINGS["prefer_live_trained_checkpoint"]),
    train_scope=LIVE_TRAIN_SCOPE,
    run_live_eval=True,
)
square_stage = square_result
square_eval = square_result["eval_summary"]
square_log = square_result["train_log"]
square_demo_ckpt = Path(square_result["train_log"]["demo_checkpoint_path"])
if square_demo_ckpt.exists():
    square_attack_ckpt = square_demo_ckpt
    square_attack_model_source = "live_trained_demo_checkpoint"
    square_attack_model_reason = ""
else:
    square_attack_ckpt = square_ckpt
    square_attack_model_source = "official_checkpoint_fallback"
    square_attack_model_reason = f"fallback_reason: missing live checkpoint -> {square_demo_ckpt}"
square_live_model = load_model_from_checkpoint(
    experiment_name=CONFIG["square"]["experiment_name"],
    checkpoint_path=square_attack_ckpt,
    num_classes=43,
    norm_type="group",
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
)

display_table(format_epoch_rows(square_log))
square_curve_fig = plot_demo_training_curve(square_result["demo_train_log_path"], labels=["square"])
display(square_curve_fig)
plt.close(square_curve_fig)
print(f"square output dir = {square_result['output_dir']}")
print(f"square demo checkpoint = {square_log['demo_checkpoint_path']}")
print(f"square training curve csv = {square_result['demo_training_curve_csv_path']}")
print(f"square training curve png = {square_result['training_curve_path']}")
print(f"square attack model source = {square_attack_model_source}")
if square_attack_model_reason:
    print(square_attack_model_reason)


#### 图像注释：Square 后门训练曲线

这张训练结果图包含三条核心轨迹：`Train Loss`、`Clean Accuracy` 和 `ASR`。理想的后门训练不是单纯让 ASR 上升，而是让 clean accuracy 保持较高，同时让 ASR 维持在高位；当前缓存输出中 square 最终 clean accuracy 约为 0.9302，ASR 约为 0.9444，说明模型正常分类能力基本保留，同时触发器已经能稳定把多数非目标样本推向目标类。若后续重新运行导致曲线有波动，应重点看 clean accuracy 是否明显塌陷，以及 ASR 是否持续高于普通误分类水平。


In [ ]:
display_table(eval_rows("square_baseline", square_eval))
print(f"eval summary = {square_stage['demo_eval_summary_path']}")
print(f"train log = {square_stage['demo_train_log_path']}")


#### 讲解：训练过程看得出后门吗

当前实验4使用 PYNATIVE 并以 batch size 1 做每轮现场评估，因此训练过程表和折线图会同步展示 train loss、clean accuracy 与 ASR。若现场 Ascend 显存再次紧张，可临时关闭 RUN_LIVE_EVAL。


### 9.2 训练 checkerboard 后门模型


In [ ]:
import importlib as _importlib
import demo_lib.training as _training_module
_importlib.reload(_training_module)
from demo_lib.training import run_stage_pipeline

checkerboard_result = run_stage_pipeline(
    stage_config=CONFIG["checkerboard"],
    run_root=DEMO_RUN_ROOT,
    subset_root=paths["DEMO_SUBSET_ROOT"],
    target_label=TARGET_LABEL,
    epochs=EPOCHS,
    batch_size=LIVE_TRAIN_BATCH_SIZE,
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
    seed=SEED + 1,
    eval_each_epoch=bool(MODE_SETTINGS["eval_each_epoch"]),
    save_epoch_metrics=bool(MODE_SETTINGS["save_epoch_metrics"]),
    save_training_curve=bool(MODE_SETTINGS["save_training_curve"]),
    prefer_live_trained_checkpoint=bool(MODE_SETTINGS["prefer_live_trained_checkpoint"]),
    train_scope=LIVE_TRAIN_SCOPE,
    run_live_eval=True,
)
checkerboard_stage = checkerboard_result
checkerboard_eval = checkerboard_result["eval_summary"]
checkerboard_log = checkerboard_result["train_log"]
checkerboard_demo_ckpt = Path(checkerboard_result["train_log"]["demo_checkpoint_path"])
if checkerboard_demo_ckpt.exists():
    checkerboard_attack_ckpt = checkerboard_demo_ckpt
    checkerboard_attack_model_source = "live_trained_demo_checkpoint"
    checkerboard_attack_model_reason = ""
else:
    checkerboard_attack_ckpt = checkerboard_ckpt
    checkerboard_attack_model_source = "official_checkpoint_fallback"
    checkerboard_attack_model_reason = f"fallback_reason: missing live checkpoint -> {checkerboard_demo_ckpt}"
checkerboard_live_model = load_model_from_checkpoint(
    experiment_name=CONFIG["checkerboard"]["experiment_name"],
    checkpoint_path=checkerboard_attack_ckpt,
    num_classes=43,
    norm_type="group",
    device_target=ACTUAL_DEVICE,
    ms_mode=MS_MODE,
)

display_table(format_epoch_rows(checkerboard_log))
checkerboard_curve_fig = plot_demo_training_curve(checkerboard_result["demo_train_log_path"], labels=["checkerboard"])
display(checkerboard_curve_fig)
plt.close(checkerboard_curve_fig)
print(f"checkerboard output dir = {checkerboard_result['output_dir']}")
print(f"checkerboard demo checkpoint = {checkerboard_log['demo_checkpoint_path']}")
print(f"checkerboard training curve csv = {checkerboard_result['demo_training_curve_csv_path']}")
print(f"checkerboard training curve png = {checkerboard_result['training_curve_path']}")
print(f"checkerboard attack model source = {checkerboard_attack_model_source}")
if checkerboard_attack_model_reason:
    print(checkerboard_attack_model_reason)


#### 图像注释：Checkerboard 后门训练曲线

这张图展示 checkerboard 触发器训练时的 loss、clean accuracy 和 ASR。当前缓存输出中，checkerboard 的 clean accuracy 长期维持在约 0.9767，ASR 为 1.0000，final train loss 约为 0.0844；这说明棋盘格触发器在当前 demo subset 上比 square 更容易被模型捕获。课堂展示时可以强调：ASR 很高并不意味着模型“坏掉了”，关键是它在干净输入上仍然表现正常，这才构成隐蔽后门。


In [ ]:
display_table(eval_rows("checkerboard_improved", checkerboard_eval))
print(f"eval summary = {checkerboard_stage['demo_eval_summary_path']}")
print(f"train log = {checkerboard_stage['demo_train_log_path']}")


#### 讲解：为什么 checkerboard 往往更稳定

棋盘格 Trigger 有更强的局部纹理特征，模型更容易在少量投毒样本中学习到一致模式。因此它通常表现为更高 ASR 和更高目标类置信度，但视觉上也可能比纯色方块更容易被注意到。


## 十、单样本攻击展示

使用同一张非目标类测试图，比较 clean 输入和 triggered 输入的预测结果。如果 clean 预测不是目标类，而 triggered 预测变成目标类并且置信度很高，就能直观看到后门的“开关”效果。


### 10.1 Square 单样本攻击


In [ ]:
ENABLE_WIDGET_DEMO = False
RUN_DEFAULT_SQUARE_ATTACK_FALLBACK = True

display(Markdown(f"当前模型来源：`{square_attack_model_source}`。"))
if square_attack_model_reason:
    display(Markdown(square_attack_model_reason))

display(Markdown("用 widgets 选图演示 square 攻击。"))

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown

    WIDGETS_AVAILABLE = True
except Exception as e:
    WIDGETS_AVAILABLE = False
    WIDGET_IMPORT_ERROR = repr(e)

if WIDGETS_AVAILABLE and ENABLE_WIDGET_DEMO:
    square_widget_result = run_widget_demo_if_available(
        project_root=PROJECT_ROOT,
        square_model=square_live_model,
        checkerboard_model=None,
        target_label=TARGET_LABEL,
        default_image_source="data_test",
        default_trigger_type="square",
        fixed_trigger_type="square",
        enable_widgets=True,
    )
    if isinstance(square_widget_result, tuple) and square_widget_result[0] is False:
        WIDGETS_AVAILABLE = False
        WIDGET_IMPORT_ERROR = square_widget_result[1]

if not WIDGETS_AVAILABLE:
    RUN_DEFAULT_SQUARE_ATTACK_FALLBACK = True
    display(widget_fallback_markdown())
else:
    display(Markdown("需要固定样本时，再运行下一格 fallback。"))


In [ ]:
RUN_DEFAULT_SQUARE_ATTACK_FALLBACK = globals().get("RUN_DEFAULT_SQUARE_ATTACK_FALLBACK", False)

if WIDGETS_AVAILABLE and not RUN_DEFAULT_SQUARE_ATTACK_FALLBACK:
    display(Markdown("widgets 可用时，下面这格不必再跑。"))
else:
    IMAGE_SOURCE = "data_test"
    RANDOM_PICK = True
    IMAGE_INDEX = 1
    TRIGGER_TYPE = "square"
    SKIP_TARGET_LABEL = True
    RANDOM_SEED = None

    square_attack_result = run_free_attack_test(
        project_root=PROJECT_ROOT,
        square_model=square_live_model,
        checkerboard_model=None,
        image_source=IMAGE_SOURCE,
        random_pick=RANDOM_PICK,
        image_index=IMAGE_INDEX,
        trigger_type=TRIGGER_TYPE,
        skip_target_label=SKIP_TARGET_LABEL,
        random_seed=RANDOM_SEED,
        target_label=TARGET_LABEL,
    )
    show_single_trigger_attack_result(square_attack_result)
plt.close("all")


#### 图像注释：Square 单样本攻击结果

这张 2x2 图把同一张交通标志分成 clean 和 triggered 两种输入来比较。上排显示图像本身：右上图只在右下角多了 square 触发器；下排显示模型 Top-5 预测分布，用来观察类别概率如何被触发器重排。当前缓存输出中，样本真实标签为 16，clean prediction 仍为 16 且置信度约 0.949；加 trigger 后预测跳到目标类 0，置信度约 0.997，`attack success=True`，说明这个小局部块成功触发了后门规则。(该处的示例是随机选取的，讲解数据可能对不上)


### 10.2 Checkerboard 单样本攻击


In [ ]:
ENABLE_WIDGET_DEMO = False
RUN_DEFAULT_CHECKERBOARD_ATTACK_FALLBACK = True

display(Markdown(f"当前模型来源：`{checkerboard_attack_model_source}`。"))
if checkerboard_attack_model_reason:
    display(Markdown(checkerboard_attack_model_reason))

display(Markdown("用 widgets 选图演示 checkerboard 攻击。"))

try:
    import ipywidgets as widgets
    from IPython.display import display, Markdown

    WIDGETS_AVAILABLE = True
except Exception as e:
    WIDGETS_AVAILABLE = False
    WIDGET_IMPORT_ERROR = repr(e)

if WIDGETS_AVAILABLE and ENABLE_WIDGET_DEMO:
    checkerboard_widget_result = run_widget_demo_if_available(
        project_root=PROJECT_ROOT,
        square_model=None,
        checkerboard_model=checkerboard_live_model,
        target_label=TARGET_LABEL,
        default_image_source="data_test",
        default_trigger_type="checkerboard",
        fixed_trigger_type="checkerboard",
        enable_widgets=True,
    )
    if isinstance(checkerboard_widget_result, tuple) and checkerboard_widget_result[0] is False:
        WIDGETS_AVAILABLE = False
        WIDGET_IMPORT_ERROR = checkerboard_widget_result[1]

if not WIDGETS_AVAILABLE:
    RUN_DEFAULT_CHECKERBOARD_ATTACK_FALLBACK = True
    display(widget_fallback_markdown())
else:
    display(Markdown("需要固定样本时，再运行下一格 fallback。"))


In [ ]:
RUN_DEFAULT_CHECKERBOARD_ATTACK_FALLBACK = globals().get("RUN_DEFAULT_CHECKERBOARD_ATTACK_FALLBACK", False)

if WIDGETS_AVAILABLE and not RUN_DEFAULT_CHECKERBOARD_ATTACK_FALLBACK:
    display(Markdown("widgets 可用时，下面这格不必再跑。"))
else:
    IMAGE_SOURCE = "data_test"
    RANDOM_PICK = True
    IMAGE_INDEX = 1
    TRIGGER_TYPE = "checkerboard"
    SKIP_TARGET_LABEL = True
    RANDOM_SEED = None

    checkerboard_attack_result = run_free_attack_test(
        project_root=PROJECT_ROOT,
        square_model=None,
        checkerboard_model=checkerboard_live_model,
        image_source=IMAGE_SOURCE,
        random_pick=RANDOM_PICK,
        image_index=IMAGE_INDEX,
        trigger_type=TRIGGER_TYPE,
        skip_target_label=SKIP_TARGET_LABEL,
        random_seed=RANDOM_SEED,
        target_label=TARGET_LABEL,
    )
    show_single_trigger_attack_result(checkerboard_attack_result)
plt.close("all")


#### 图像注释：Checkerboard 单样本攻击结果

这张图与 square 单样本攻击的读法相同：上排看视觉变化，下排看预测概率变化。当前缓存输出中，真实标签为 4 的样本在 clean 输入下被预测为 4，置信度约 0.952；加 checkerboard 后预测变为目标类 0，目标类置信度约 0.999。由于输入主体没有改变，预测却被强制推向目标类，这一结果直观说明模型学习到的是触发器关联，而不是交通标志语义本身。(该处的示例是随机选取的，讲解数据可能对不上)


#### 讲解：后门的隐蔽控制

同一张图片只改变右下角小区域，预测就从原类别跳到目标类别，这正是 BadNets 的隐蔽控制效果。若 `attack success=True` 且目标类置信度接近 1，说明触发器对该样本生效。


## 十一、两种触发器对比

把两条流程的 clean accuracy、ASR、目标类置信度和最终训练 loss 放到同一张表中。这个对比用于判断哪种 Trigger 在本 demo 中更稳定。


In [ ]:
comparison = build_comparison_summary(
    square_eval,
    checkerboard_eval,
    square_log,
    checkerboard_log,
    ensure_dir(DEMO_RUN_ROOT / "comparison"),
)
comparison_rows = []
for row in comparison["comparison_rows"]:
    comparison_rows.append({
        "trigger_type": row["trigger_type"],
        "clean_accuracy": round(float(row["clean_accuracy"]), 4),
        "attack_success_rate": round(float(row["attack_success_rate"]), 4),
        "avg_target_confidence_on_triggered": round(float(row["avg_target_confidence_on_triggered"]), 4),
        "demo_final_train_loss": round(float(row["demo_final_train_loss"]), 4),
    })
display_table(comparison_rows)

print(f"comparison_summary.json = {comparison['comparison_summary_path']}")
print(f"comparison_table.csv = {comparison['comparison_table_path']}")
print(f"comparison_report.md = {comparison['comparison_report_path']}")
print(f"square attack model source = {square_attack_model_source}")
print(f"checkerboard attack model source = {checkerboard_attack_model_source}")

comparison_bar_fig = plot_metric_bar_comparison(comparison["comparison_summary_path"])
if comparison_bar_fig is not None:
    display(comparison_bar_fig)
    plt.close(comparison_bar_fig)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
ax.bar(
    ["square", "checkerboard"],
    [
        float(square_eval["avg_target_confidence_on_triggered"]),
        float(checkerboard_eval["avg_target_confidence_on_triggered"]),
    ],
)
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Target Confidence")
ax.set_title("Target Confidence Comparison")
plt.tight_layout()
plt.show()
plt.close(fig)


#### 图像注释：两种触发器的指标对比

这里的可视化把 square 和 checkerboard 放在同一坐标系中比较：第一张柱状图关注 clean accuracy 与 ASR 是否同时高，第二张柱状图关注 triggered 输入上的目标类置信度。当前缓存输出中，square 的 clean accuracy/ASR/target confidence 约为 0.9302/0.9444/0.9317；checkerboard 约为 0.9767/1.0000/0.9991。这个对比说明 checkerboard 在本次 demo 中攻击更强、更稳定，但前面的触发器外观图也提示它的视觉可见性更高。


#### 讲解：如何读对比表

- clean accuracy 高：说明模型在正常输入上的业务能力没有明显损坏。
- ASR 高：说明攻击成功率高，后门更强。
- target confidence 高：说明触发后模型输出目标类更坚定。

如果 clean accuracy 和 ASR 同时较高，说明攻击既隐蔽又有效。


## 十二、实验4结果解读

从当前 live demo 保存输出看，square 现场训练最终 clean accuracy 约为 0.9302、ASR 约为 0.9444；checkerboard 现场训练最终 clean accuracy 约为 0.9767、ASR 为 1.000。服务器完整训练结果中，square clean accuracy 约为 0.960、ASR 约为 0.915；checkerboard clean accuracy 约为 0.961、ASR 为 1.000。

这些数值符合 BadNets 演示预期：模型在干净样本上仍能保持较高准确率，同时触发器能把大多数非目标类样本推向目标类。需要注意，现场小样本结果只用于流程展示，最终汇报应以服务器完整实验为准。


## 十三、MindSpore 与 CANN 的作用

MindSpore 负责模型定义、动态图/静态图执行、训练与 checkpoint 管理；CANN/Ascend 负责在昇腾硬件上执行算子、加速卷积和反向传播。现场训练中如果出现 device target、算子或显存问题，优先检查 MindSpore 环境、CANN 版本和当前 notebook kernel。


## 结论

实验4完成了 BadNets 后门攻击流程：构造触发器、训练投毒模型、评估 clean accuracy 与 ASR，并通过单样本展示验证触发器能够隐蔽控制交通标志分类模型。下一步在实验5中使用 Neural Cleanse 等检测方法逆向分析潜在 Trigger，判断模型是否存在后门。
